In [ ]:
cd /content/drive/MyDrive/HybridRE/Data

/content/drive/MyDrive/HybridRE/Data


In [ ]:
import pandas as pd
import scorer
from contextlib import redirect_stdout
import io

THRESHOLD = 0.90


def silent_score(gold, pred):
    """
    Call scorer.score without any internal printing
    """
    f = io.StringIO()
    with redirect_stdout(f):
        p, r, f1 = scorer.score(gold, pred, verbose=False)
    return p, r, f1


def evaluate_file(csv_path):
    print("=" * 80)
    print(f"Evaluating file: {csv_path}")
    print("=" * 80)

    df = pd.read_csv(csv_path)

    gold = df["True_Labels"].tolist()
    plm_pred = df["Initial_Predictions"].tolist()
    llm_pred = df["LLM_Prediction"].tolist()
    conf = df["Confidence"].tolist()

    # ---------------------------
    # 1. PLM ONLY
    # ---------------------------
    p_plm, r_plm, f1_plm = silent_score(gold, plm_pred)
    print("\n[PLM ONLY]")
    print(f"Precision: {p_plm:.3%}")
    print(f"Recall:    {r_plm:.3%}")
    print(f"F1:        {f1_plm:.3%}")

    # ---------------------------
    # 2. LLM ONLY
    # ---------------------------
    p_llm, r_llm, f1_llm = silent_score(gold, llm_pred)
    print("\n[LLM ONLY]")
    print(f"Precision: {p_llm:.3%}")
    print(f"Recall:    {r_llm:.3%}")
    print(f"F1:        {f1_llm:.3%}")

    # ---------------------------
    # 3. HYBRID RE (threshold)
    # ---------------------------
    # hybrid_pred = []
    # for c, p_plm_pred, p_llm_pred in zip(conf, plm_pred, llm_pred):
    #     if c >= THRESHOLD:
    #         hybrid_pred.append(p_plm_pred)
    #     else:
    #         hybrid_pred.append(p_llm_pred)


    hybrid_pred = []
    for c, p_plm_pred, p_llm_pred in zip(conf, plm_pred, llm_pred):

        # Only reclassify positive PLM predictions
        if p_plm_pred != "no_relation" and c < THRESHOLD:
            hybrid_pred.append(p_llm_pred)
        else:
            hybrid_pred.append(p_plm_pred)


    p_h, r_h, f1_h = silent_score(gold, hybrid_pred)

    delta_f1 = f1_h - f1_plm

    print(f"\n[HYBRID RE | threshold = {THRESHOLD}]")
    print(f"Precision: {p_h:.3%}")
    print(f"Recall:    {r_h:.3%}")
    print(f"F1:        {f1_h:.3%}")
    print(f"ΔF1 (Hybrid − PLM): {delta_f1:+.3%}")


In [ ]:
from pathlib import Path
# =========================
# BATCH EVALUATION
# =========================
BASE_DIR = Path("3_Final_predictions")

PLMS = [
    "PLM_ROBERTA-LARGE"
]

DATASETS = ["TACREV"]
SPLITS = ["TEST"]

for dataset in DATASETS:
    folder = BASE_DIR / dataset
    if not folder.exists():
        continue

    print("\n" + "#" * 90)
    print(f"### DATASET: {dataset}")
    print("#" * 90)

    for plm in PLMS:
        print("\n" + "-" * 70)
        print(f"PLM: {plm}")
        print("-" * 70)

        files = sorted(folder.glob(f"{plm}_{dataset}_*_LORA_*.csv"))

        if not files:
            print("No files found.")
            continue

        for f in files:
            evaluate_file(f)


##########################################################################################
### DATASET: TACREV
##########################################################################################

----------------------------------------------------------------------
PLM: PLM_ROBERTA-LARGE
----------------------------------------------------------------------
Evaluating file: 3_Final_predictions/TACREV/PLM_ROBERTA-LARGE_TACREV_DEV_LORA_LAMA.csv

[PLM ONLY]
Precision: 79.202%
Recall:    85.606%
F1:        82.279%

[LLM ONLY]
Precision: 82.845%
Recall:    86.969%
F1:        84.857%

[HYBRID RE | threshold = 0.9]
Precision: 86.669%
Recall:    82.245%
F1:        84.399%
ΔF1 (Hybrid − PLM): +2.119%
Evaluating file: 3_Final_predictions/TACREV/PLM_ROBERTA-LARGE_TACREV_DEV_LORA_Mistral.csv

[PLM ONLY]
Precision: 77.791%
Recall:    85.321%
F1:        81.382%

[LLM ONLY]
Precision: 83.134%
Recall:    86.396%
F1:        84.734%

[HYBRID RE | threshold = 0.9]
Precision: 86.843%
Recall:    8